# Проверка daily flow классификации претензий

Этот notebook запускает готовую логику по шагам, чтобы было видно промежуточные результаты и где именно может зависнуть прогон.

По умолчанию notebook:
- берёт конфиг `configs/auto_loan.json`;
- читает данные за вчера относительно `SCORE_DATE`;
- ограничивает обработку `SAMPLE_ROWS`, чтобы быстро проверить LLM;
- не пишет результат в Oracle, пока `WRITE_TO_ORACLE = False`.

In [ ]:
from __future__ import annotations

import json
import time
from datetime import date
from pathlib import Path

import pandas as pd
from IPython.display import display

from complaints_scoring_flow import _merge_settings
from llm_pipeline import (
    build_classification_model,
    build_judge_issues_chain,
    build_judge_requests_chain,
    build_llm_from_config,
    build_stage1_chain,
    run_judge,
    run_stage1_classification,
)
from tasks.scoring_functions import _build_llm_config, _read_product_context, _resolve_config_path, create_oracle_engine, read_source_data
from utils.result_builder import build_result_frame, compute_score_window, filter_source_data
from utils.sql_writer import ClassificationSQLWriter

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 250)

## 1. Параметры проверки

`SAMPLE_ROWS` ограничивает число претензий, которые реально уйдут в LLM. Для первого запуска лучше оставить 3-5 строк.

In [ ]:
CONFIG_PATH = Path("configs/auto_loan.json")
SETTINGS_PATH = Path("settings.json")

SCORE_DATE = date.today().isoformat()  # можно заменить на "2026-06-10"
LOOKBACK_DAYS = 1
SAMPLE_ROWS = 5
WRITE_TO_ORACLE = False

print("CONFIG_PATH:", CONFIG_PATH.resolve())
print("SETTINGS_PATH:", SETTINGS_PATH.resolve())
print("SCORE_DATE:", SCORE_DATE)
print("LOOKBACK_DAYS:", LOOKBACK_DAYS)
print("SAMPLE_ROWS:", SAMPLE_ROWS)
print("WRITE_TO_ORACLE:", WRITE_TO_ORACLE)

## 2. Загрузка настроек и конфига классификатора

In [ ]:
with open(SETTINGS_PATH, encoding="utf-8") as f:
    settings = json.load(f)
with open(CONFIG_PATH, encoding="utf-8") as f:
    classifier_config = json.load(f)

config = _merge_settings(settings, classifier_config)
config["_config_dir"] = str(CONFIG_PATH.parent.resolve())

start_date, end_date = compute_score_window(SCORE_DATE, lookback_days=LOOKBACK_DAYS)

print("classifier_name:", config["classifier_name"])
print("product:", config["product"])
print("theme:", config.get("theme"))
print("category:", config.get("category"))
print("Окно данных:", f"[{start_date}, {end_date})")
display(pd.DataFrame([classifier_config]))

## 3. Чтение данных из Oracle

Если ячейка выполняется долго, значит время уходит на SQL/Oracle, а не на LLM.

In [ ]:
t0 = time.perf_counter()
print("Создаю Oracle engine...")
engine = create_oracle_engine(config)
print(f"Oracle engine создан за {time.perf_counter() - t0:.1f} сек.")

query_path = _resolve_config_path(config, "query_path", "../queries/get_data.sql")
print("SQL:", query_path)
print("Запускаю чтение данных...")
t0 = time.perf_counter()
source_df = read_source_data(engine, query_path, start_date, end_date)
print(f"Данные прочитаны за {time.perf_counter() - t0:.1f} сек. Строк: {len(source_df)}")
display(source_df.head(10))

## 4. Фильтр по продукту / теме / категории и sample для LLM

In [ ]:
filtered_df = filter_source_data(
    source_df,
    product=config["product"],
    theme=config.get("theme"),
    category=config.get("category"),
)
sample_df = filtered_df.head(SAMPLE_ROWS).copy()

print("Строк после фильтра:", len(filtered_df))
print("Строк в sample для LLM:", len(sample_df))
display(sample_df[["claim_num", "created", "product", "theme", "category", config.get("text_column", "description_claim")]].head(SAMPLE_ROWS))

## 5. Сборка LLM chains

Эта ячейка только готовит промпты, справочники и LLM-клиент. Вызовов LLM здесь ещё нет.

In [ ]:
text_column = config.get("text_column", "description_claim")
prompts_dir = _resolve_config_path(config, "prompts_dir", "../prompts")
taxonomy_issues_path = _resolve_config_path(config, "taxonomy_issues_path")
taxonomy_requests_path = _resolve_config_path(config, "taxonomy_requests_path")

print("prompts_dir:", prompts_dir)
print("taxonomy_issues_path:", taxonomy_issues_path)
print("taxonomy_requests_path:", taxonomy_requests_path)

issues_table = taxonomy_issues_path.read_text(encoding="utf-8")
requests_table = taxonomy_requests_path.read_text(encoding="utf-8")
product_context = _read_product_context(config)

print("Длина product_context:", len(product_context))
print("Issues taxonomy labels:", issues_table.count('"sub_category"'))
print("Requested actions taxonomy labels:", requests_table.count('"sub_category"'))

llm = build_llm_from_config(_build_llm_config(config))
output_model = build_classification_model(taxonomy_issues_path, taxonomy_requests_path)
classification_chain = build_stage1_chain(
    llm,
    prompts_dir / "stage1_classification.txt",
    issues_table,
    requests_table,
    product_context=product_context,
    output_model=output_model,
)
judge_issues_chain = build_judge_issues_chain(
    llm,
    prompts_dir / "judge_issues.txt",
    issues_table,
    product_context=product_context,
)
judge_requests_chain = build_judge_requests_chain(
    llm,
    prompts_dir / "judge_requests.txt",
    requests_table,
    product_context=product_context,
)
print("Chains готовы")

## 6. Классификация sample

Если эта ячейка выполняется долго, время уходит на LLM-классификацию.

In [ ]:
runtime = config.get("runtime", {})

if sample_df.empty:
    classified_df = pd.DataFrame()
    print("Sample пустой, классификацию пропускаю")
else:
    print("Запускаю LLM-классификацию...")
    t0 = time.perf_counter()
    classified_df = run_stage1_classification(
        sample_df,
        classification_chain,
        text_column=text_column,
        batch_size=runtime.get("batch_size", 100),
        max_concurrency=runtime.get("max_concurrency", 5),
        retries=runtime.get("retries", 4),
        base_sleep_seconds=runtime.get("backoff_base_seconds", 2.0),
    )
    print(f"Классификация завершена за {time.perf_counter() - t0:.1f} сек. Строк: {len(classified_df)}")

display(classified_df[["claim_num", "issues_pred", "requested_actions_pred"]].head(SAMPLE_ROWS) if not classified_df.empty else classified_df)

## 7. Judge по issues и requested_actions

Если эта ячейка выполняется долго, время уходит на LLM-judge.

In [ ]:
if classified_df.empty:
    judge_issues_df = pd.DataFrame()
    judge_requests_df = pd.DataFrame()
    print("Нет классифицированных строк, judge пропускаю")
else:
    print("Запускаю judge issues...")
    t0 = time.perf_counter()
    judge_issues_df = run_judge(
        classified_df,
        judge_issues_chain,
        labels_column="issues_pred",
        text_column=text_column,
        batch_size=runtime.get("batch_size", 100),
        max_concurrency=runtime.get("max_concurrency", 5),
        retries=runtime.get("retries", 4),
        base_sleep_seconds=runtime.get("backoff_base_seconds", 2.0),
    )
    print(f"Judge issues завершён за {time.perf_counter() - t0:.1f} сек. Строк: {len(judge_issues_df)}")

    print("Запускаю judge requested_actions...")
    t0 = time.perf_counter()
    judge_requests_df = run_judge(
        classified_df,
        judge_requests_chain,
        labels_column="requested_actions_pred",
        text_column=text_column,
        batch_size=runtime.get("batch_size", 100),
        max_concurrency=runtime.get("max_concurrency", 5),
        retries=runtime.get("retries", 4),
        base_sleep_seconds=runtime.get("backoff_base_seconds", 2.0),
    )
    print(f"Judge requested_actions завершён за {time.perf_counter() - t0:.1f} сек. Строк: {len(judge_requests_df)}")

print("judge_issues_df")
display(judge_issues_df.head(20))
print("judge_requests_df")
display(judge_requests_df.head(20))

## 8. Сбор финальной таблицы результата

In [ ]:
if classified_df.empty:
    result_df = pd.DataFrame()
else:
    result_df = build_result_frame(
        classified_df,
        judge_issues_df,
        judge_requests_df,
        score_date=SCORE_DATE,
        classifier_name=config["classifier_name"],
    )

print("Строк результата:", len(result_df))
display(result_df.head(50))

## 9. Опциональная запись результата в Oracle

По умолчанию запись выключена. Чтобы проверить запись, поставьте `WRITE_TO_ORACLE = True` в первой ячейке параметров.

In [ ]:
if WRITE_TO_ORACLE:
    print("Пишу результат в Oracle...")
    t0 = time.perf_counter()
    writer = ClassificationSQLWriter(
        result_df,
        config.get("output_table", "ema_complaints_classification"),
        config["classifier_name"],
        SCORE_DATE,
    )
    writer.write_data(engine)
    print(f"Запись завершена за {time.perf_counter() - t0:.1f} сек. Строк: {len(result_df)}")
else:
    print("WRITE_TO_ORACLE = False, запись пропущена")